In [1]:
import os

from dotenv import load_dotenv
from groq import Groq
from langchain_groq import ChatGroq

load_dotenv()

DEFAULT_GROQ_CHAT_MODELS = [
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
    "mixtral-8x7b-32768",
]


def read_env(name: str) -> str:
    return os.getenv(name, "").strip()


def get_available_groq_models(api_key: str) -> list[str]:
    client = Groq(api_key=api_key)
    models = sorted(
        model.id
        for model in client.models.list().data
        if getattr(model, "active", True)
    )
    if not models:
        raise RuntimeError("Groq returned no active models for this API key.")
    return models


def filter_chat_models(models: list[str]) -> list[str]:
    blocked_terms = ("whisper", "tts", "speech")
    return [
        model_name
        for model_name in models
        if not any(term in model_name.lower() for term in blocked_terms)
    ]


def select_groq_model(chat_models: list[str], requested_model: str) -> str:
    if not chat_models:
        raise RuntimeError("Groq returned no chat-capable models for this API key.")

    if requested_model:
        if requested_model not in chat_models:
            raise ValueError(
                f"GROQ_MODEL '{requested_model}' is not available for this API key. "
                f"Choose one of: {chat_models}"
            )
        return requested_model

    return next(
        (model_name for model_name in DEFAULT_GROQ_CHAT_MODELS if model_name in chat_models),
        chat_models[0],
    )


groq_api_key = read_env("GROQ_API_KEY")
requested_model = read_env("GROQ_MODEL")

if not groq_api_key:
    raise ValueError(
        "Set GROQ_API_KEY in your environment or .env file before running this notebook."
    )

available_models = get_available_groq_models(groq_api_key)
available_chat_models = filter_chat_models(available_models)
model_name = select_groq_model(available_chat_models, requested_model)

os.environ["GROQ_API_KEY"] = groq_api_key
llm = ChatGroq(model=model_name, temperature=0.2)

print("Available Groq chat models:", available_chat_models)
print("Using Groq model:", model_name)

Available Groq chat models: ['allam-2-7b', 'canopylabs/orpheus-arabic-saudi', 'canopylabs/orpheus-v1-english', 'groq/compound', 'groq/compound-mini', 'llama-3.1-8b-instant', 'llama-3.3-70b-versatile', 'meta-llama/llama-4-scout-17b-16e-instruct', 'meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'openai/gpt-oss-safeguard-20b', 'qwen/qwen3-32b']
Using Groq model: llama-3.3-70b-versatile


In [10]:
import os

import torch

from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoModel, AutoTokenizer

CHUNK_SIZE = 240
CHUNK_OVERLAP = 50
CHUNK_PREVIEW_LIMIT = 8
VECTOR_K = 5
EMBEDDING_MODEL_NAME = os.getenv(
    "EMBEDDING_MODEL",
    "sentence-transformers/all-MiniLM-L6-v2",
)
EMBEDDING_BATCH_SIZE = 8
EMBEDDING_DEVICE = "cpu"

RAW_SOURCES = [
    {
        "source": "finance_schema_reference.txt",
        "topic": "exact_schema",
        "text": """
The finance warehouse publishes a table named fact_invoice_aging.
The primary customer key is cust_id and the invoice key is invoice_id.
The invoice aging snapshot column is snapshot_date and the overdue bucket column is overdue_bucket.
The AR_90_PLUS field stores invoices that are more than 90 days overdue.
The collections_priority_code field identifies which overdue accounts should be worked first.
Analysts often search for fact_invoice_aging, cust_id, invoice_id, AR_90_PLUS, and collections_priority_code as exact terms.
"""
    },
    {
        "source": "finance_mart_contract.txt",
        "topic": "exact_schema",
        "text": """
The mart.finance.invoice_aging_daily table is refreshed by job_7841 every morning.
The dataset includes snapshot_date, region_code, overdue_bucket, invoice_amount, and collections_priority_code.
The downstream dashboard named dashboard_ar_watchlist reads from mart.finance.invoice_aging_daily.
If a user asks which job refreshes the invoice aging mart, the exact identifier is job_7841.
If a user asks for the bucket representing very old receivables, the exact field is AR_90_PLUS.
"""
    },
    {
        "source": "collections_glossary.txt",
        "topic": "semantic_glossary",
        "text": """
Collections teams talk about chronic late payers when customers repeatedly miss due dates and keep balances unpaid for long periods.
Business users may describe AR_90_PLUS as the overdue bucket for customers who are very late on payments.
The aging mart helps analysts find accounts with old unpaid balances, delinquent invoices, and customers at risk of slow payment.
People often ask about late-paying customers without knowing the database field names or job identifiers.
"""
    },
    {
        "source": "collections_playbook.txt",
        "topic": "business_questions",
        "text": """
Collectors review long-outstanding invoices, broken promise-to-pay cases, and customers with repeated payment delays.
Business teams ask which accounts are slipping into severe delinquency and which customers should be escalated first.
These questions are often phrased semantically rather than with exact warehouse field names.
Hybrid retrieval works well when a query mixes business phrasing with identifiers such as AR_90_PLUS or job_7841.
"""
    },
    {
        "source": "ops_runbook.txt",
        "topic": "pipeline_ops",
        "text": """
If mart.finance.invoice_aging_daily is stale, first check whether job_7841 finished successfully.
If the refresh failed, inspect the raw landing path s3://lake/raw/ar_aging/ and the transformation logs for missing partitions.
The common validation step is to compare invoice counts by region_code before and after the load.
Exact identifiers such as job_7841, mart.finance.invoice_aging_daily, and dashboard_ar_watchlist are useful for keyword retrieval.
"""
    },
    {
        "source": "data_quality_checks.txt",
        "topic": "pipeline_ops",
        "text": """
The finance quality checklist verifies that overdue_bucket totals match the aging report and that region_code counts reconcile with the source extract.
When the AR_90_PLUS totals drift, analysts inspect snapshot_date freshness, duplicate invoice_id values, and delayed refresh jobs.
This document combines exact technical terms with operational language about stale reports and aging snapshots.
"""
    },
    {
        "source": "executive_summary.txt",
        "topic": "business_questions",
        "text": """
Leaders ask which customers are most likely to remain unpaid, which balances have stayed open the longest, and which overdue accounts need fast follow-up.
They do not usually ask for column names; instead they ask about risky customers, severe delinquency, and late-payment behavior.
Dense retrieval is often strong for these paraphrased questions, while BM25 is stronger when the user includes exact field or job names.
"""
    },
]


class HFTransformerEmbeddings(Embeddings):
    def __init__(
        self,
        model_name: str = EMBEDDING_MODEL_NAME,
        device: str = EMBEDDING_DEVICE,
        batch_size: int = EMBEDDING_BATCH_SIZE,
    ):
        self.model_name = model_name
        self.device = device
        self.batch_size = batch_size
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

    def _mean_pool(self, token_embeddings, attention_mask):
        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        pooled = (token_embeddings * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        return torch.nn.functional.normalize(pooled, p=2, dim=1)

    def _encode_texts(self, texts: list[str]) -> list[list[float]]:
        vectors = []
        for start in range(0, len(texts), self.batch_size):
            batch = texts[start:start + self.batch_size]
            encoded = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            )
            encoded = {key: value.to(self.device) for key, value in encoded.items()}
            with torch.no_grad():
                model_output = self.model(**encoded)
            batch_vectors = self._mean_pool(
                model_output.last_hidden_state,
                encoded["attention_mask"],
            )
            vectors.extend(batch_vectors.cpu().tolist())
        return vectors

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self._encode_texts(texts)

    def embed_query(self, text: str) -> list[float]:
        return self._encode_texts([text])[0]


def build_chunks(
    raw_sources: list[dict],
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
) -> list[dict]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    split_documents = splitter.create_documents(
        texts=[item["text"].strip() for item in raw_sources],
        metadatas=[
            {"source": item["source"], "topic": item["topic"]}
            for item in raw_sources
        ],
    )
    return [
        {
            "source": doc.metadata["source"],
            "context": doc.page_content,
            "topic": doc.metadata["topic"],
            "chunk_id": index,
        }
        for index, doc in enumerate(split_documents, start=1)
    ]


def build_documents(chunk_rows: list[dict]) -> list[Document]:
    return [
        Document(
            page_content=chunk["context"].strip(),
            metadata={
                "id": index,
                "source": chunk["source"],
                "topic": chunk.get("topic", "unknown"),
            },
        )
        for index, chunk in enumerate(chunk_rows, start=1)
    ]


def build_embedding_backend():
    try:
        from langchain_huggingface import HuggingFaceEmbeddings

        backend = HuggingFaceEmbeddings(
            model_name=EMBEDDING_MODEL_NAME,
            model_kwargs={"device": EMBEDDING_DEVICE},
            encode_kwargs={"normalize_embeddings": True},
        )
        return backend, f"langchain-huggingface:{EMBEDDING_MODEL_NAME}"
    except Exception as exc:
        print(
            "LangChain HuggingFaceEmbeddings is unavailable in this environment; "
            f"using transformers fallback instead: {exc}"
        )
        backend = HFTransformerEmbeddings(
            model_name=EMBEDDING_MODEL_NAME,
            device=EMBEDDING_DEVICE,
            batch_size=EMBEDDING_BATCH_SIZE,
        )
        return backend, f"transformers:{EMBEDDING_MODEL_NAME}"


def build_vector_components(documents: list[Document]):
    embeddings, backend_name = build_embedding_backend()
    store = FAISS.from_documents(documents, embeddings)
    retriever = store.as_retriever(
        search_kwargs={"k": min(VECTOR_K, len(documents))}
    )
    return embeddings, store, retriever, backend_name


def preview_chunks(chunk_rows: list[dict], limit: int = CHUNK_PREVIEW_LIMIT) -> None:
    print("\nChunk preview:")
    for chunk in chunk_rows[:limit]:
        print(f"[{chunk['chunk_id']}] {chunk['source']} | topic={chunk['topic']}")
        print(chunk["context"])
        print("-" * 80)


chunks = build_chunks(RAW_SOURCES)
lc_documents = build_documents(chunks)
embedding_model, vector_store, vector_retriever, embedding_backend_name = build_vector_components(lc_documents)

print("Created chunk count:", len(chunks))
print("Topics:", sorted({chunk["topic"] for chunk in chunks}))
print("Vector backend:", embedding_backend_name)
print("FAISS documents loaded:", vector_store.index.ntotal)
preview_chunks(chunks)
print("\nCell 2 complete: chunks and FAISS are ready for retrieval.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Created chunk count: 18
Topics: ['business_questions', 'exact_schema', 'pipeline_ops', 'semantic_glossary']
Vector backend: langchain-huggingface:sentence-transformers/all-MiniLM-L6-v2
FAISS documents loaded: 18

Chunk preview:
[1] finance_schema_reference.txt | topic=exact_schema
The finance warehouse publishes a table named fact_invoice_aging.
The primary customer key is cust_id and the invoice key is invoice_id.
The invoice aging snapshot column is snapshot_date and the overdue bucket column is overdue_bucket.
--------------------------------------------------------------------------------
[2] finance_schema_reference.txt | topic=exact_schema
The AR_90_PLUS field stores invoices that are more than 90 days overdue.
The collections_priority_code field identifies which overdue accounts should be worked first.
--------------------------------------------------------------------------------
[3] finance_schema_reference.txt | topic=exact_schema
Analysts often search for fact_invoice_aging

In [11]:
if "llm" not in globals() or "lc_documents" not in globals() or "vector_retriever" not in globals():
    raise RuntimeError("Run cells 1 and 2 first.")

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.retrievers import BM25Retriever

BM25_K = 5
HYBRID_K = 4
RRF_K = 60
FETCH_PRINT_LIMIT = 3

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a RAG assistant.
Answer only from the retrieved context.
If the context is insufficient, say you do not know.
"""
    ),
    (
        "human",
        """
Question:
{question}

Retrieved context:
{context}

Give a concise grounded answer.
"""
    ),
])

document_lookup = {int(doc.metadata["id"]): doc for doc in lc_documents}
bm25_retriever = BM25Retriever.from_documents(lc_documents)
bm25_retriever.k = min(BM25_K, len(lc_documents))


def reciprocal_rank_fusion(
    ranked_lists: list[list[Document]],
    document_lookup: dict[int, Document],
    k: int = RRF_K,
) -> list[Document]:
    fused_scores = {}
    for ranked_docs in ranked_lists:
        for rank, doc in enumerate(ranked_docs, start=1):
            doc_id = int(doc.metadata["id"])
            fused_scores[doc_id] = fused_scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return [
        document_lookup[doc_id]
        for doc_id, _ in sorted(fused_scores.items(), key=lambda item: item[1], reverse=True)
    ]


def ranked_rows(retrieved_docs: list[Document]) -> list[dict]:
    rows = []
    for rank, doc in enumerate(retrieved_docs, start=1):
        rows.append({
            "rank": rank,
            "id": int(doc.metadata["id"]),
            "source": doc.metadata["source"],
            "topic": doc.metadata.get("topic", "unknown"),
            "context": doc.page_content,
        })
    return rows


def build_context_block(retrieved_docs: list[Document]) -> str:
    return "\n\n".join(
        f"Source: {doc.metadata['source']}\nContext: {doc.page_content}"
        for doc in retrieved_docs
    )


def answer_from_docs(query: str, retrieved_docs: list[Document]) -> str:
    response = (ANSWER_PROMPT | llm).invoke({
        "question": query,
        "context": build_context_block(retrieved_docs),
    })
    return response.content if hasattr(response, "content") else str(response)


def print_ranked_chunks(title: str, rows: list[dict], limit: int | None = None) -> None:
    visible_rows = rows if limit is None else rows[:limit]
    print(f"\n{title}:")
    if limit is not None and len(rows) > limit:
        print(f"Showing top {limit} of {len(rows)} results")
    for row in visible_rows:
        print(f"[{row['rank']}] {row['source']} | topic={row['topic']}")
        print(row["context"])
        print("-" * 80)


user_query = "How can I identify chronic late payers, and which job refreshes the invoice aging mart?"
semantic_fetch = vector_retriever.invoke(user_query)
bm25_fetch = bm25_retriever.invoke(user_query)
rrf_docs = reciprocal_rank_fusion(
    [semantic_fetch, bm25_fetch],
    document_lookup=document_lookup,
 )[:HYBRID_K]
rrf_context = build_context_block(rrf_docs)
hybrid_answer = answer_from_docs(user_query, rrf_docs)

hybrid_result = {
    "query": user_query,
    "semantic_docs": semantic_fetch,
    "bm25_docs": bm25_fetch,
    "rrf_docs": rrf_docs,
    "rrf_context": rrf_context,
    "hybrid_answer": hybrid_answer,
}

semantic_rows = ranked_rows(semantic_fetch)
bm25_rows = ranked_rows(bm25_fetch)
rrf_rows = ranked_rows(rrf_docs)

print("Vector backend:", embedding_backend_name)
print("Query:", user_query)
print_ranked_chunks("Semantic fetch", semantic_rows, limit=FETCH_PRINT_LIMIT)
print_ranked_chunks("BM25 fetch", bm25_rows, limit=FETCH_PRINT_LIMIT)
print_ranked_chunks("RRF fused", rrf_rows)
print("\nContext passed to LLM after RRF:\n")
print(rrf_context)
print("\nLLM answer:\n", hybrid_answer)

Vector backend: langchain-huggingface:sentence-transformers/all-MiniLM-L6-v2
Query: How can I identify chronic late payers, and which job refreshes the invoice aging mart?

Semantic fetch:
Showing top 3 of 5 results
[1] collections_glossary.txt | topic=semantic_glossary
The aging mart helps analysts find accounts with old unpaid balances, delinquent invoices, and customers at risk of slow payment.
People often ask about late-paying customers without knowing the database field names or job identifiers.
--------------------------------------------------------------------------------
[2] finance_mart_contract.txt | topic=exact_schema
The mart.finance.invoice_aging_daily table is refreshed by job_7841 every morning.
The dataset includes snapshot_date, region_code, overdue_bucket, invoice_amount, and collections_priority_code.
--------------------------------------------------------------------------------
[3] collections_glossary.txt | topic=semantic_glossary
Collections teams talk about c

In [12]:
if "hybrid_result" not in globals() or "answer_from_docs" not in globals():
    raise RuntimeError("Run cells 1, 2, and 3 first.")

query = hybrid_result["query"]
semantic_docs = hybrid_result["semantic_docs"]
rrf_docs = hybrid_result["rrf_docs"]

dense_answer = answer_from_docs(query, semantic_docs[:HYBRID_K])
hybrid_answer = hybrid_result["hybrid_answer"]

comparison_result = {
    "query": query,
    "dense_answer": dense_answer,
    "hybrid_answer": hybrid_answer,
    "semantic_docs": semantic_docs,
    "rrf_docs": rrf_docs,
}

print("Query:", query)
print("\nDense-only answer:\n", dense_answer)
print("\nHybrid answer:\n", hybrid_answer)

Query: How can I identify chronic late payers, and which job refreshes the invoice aging mart?

Dense-only answer:
 To identify chronic late payers, you can look for customers with repeated overdue invoices, particularly those in the AR_90_PLUS overdue bucket, which indicates very late payments. The job that refreshes the invoice aging mart is job_7841.

Hybrid answer:
 To identify chronic late payers, you can look for customers in the AR_90_PLUS overdue bucket, which indicates they are very late on payments. The job that refreshes the invoice aging mart is job_7841.


In [13]:
if "comparison_result" not in globals():
    raise RuntimeError("Run cells 1 through 4 first.")

from langchain_core.prompts import ChatPromptTemplate

GROUND_TRUTH_ANSWER = (
    "To identify chronic late payers, use the invoice aging mart to find customers with "
    "old unpaid balances, delinquent invoices, and severe overdue buckets such as AR_90_PLUS. "
    "The job that refreshes the invoice aging mart is job_7841."
)

JUDGE_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are an evaluation judge for RAG answers.
Compare two answers against the provided ground truth and decide which is closer.
Prefer factual correctness, completeness, and grounding over style.
"""
    ),
    (
        "human",
        """
Query:
{query}

Ground truth:
{ground_truth}

Dense-only answer:
{dense_answer}

Hybrid answer:
{hybrid_answer}

Return exactly this format:
Winner: <dense|hybrid|tie>
Reasoning: <short explanation>
"""
    ),
])

judge_response = (JUDGE_PROMPT | llm).invoke({
    "query": comparison_result["query"],
    "ground_truth": GROUND_TRUTH_ANSWER,
    "dense_answer": comparison_result["dense_answer"],
    "hybrid_answer": comparison_result["hybrid_answer"],
})
judge_text = judge_response.content if hasattr(judge_response, "content") else str(judge_response)

print("Query:", comparison_result["query"])
print("\nGround truth:\n", GROUND_TRUTH_ANSWER)
print("\nDense-only answer:\n", comparison_result["dense_answer"])
print("\nHybrid answer:\n", comparison_result["hybrid_answer"])
print("\nLLM judge result:\n", judge_text)

Query: How can I identify chronic late payers, and which job refreshes the invoice aging mart?

Ground truth:
 To identify chronic late payers, use the invoice aging mart to find customers with old unpaid balances, delinquent invoices, and severe overdue buckets such as AR_90_PLUS. The job that refreshes the invoice aging mart is job_7841.

Dense-only answer:
 To identify chronic late payers, you can look for customers with repeated overdue invoices, particularly those in the AR_90_PLUS overdue bucket, which indicates very late payments. The job that refreshes the invoice aging mart is job_7841.

Hybrid answer:
 To identify chronic late payers, you can look for customers in the AR_90_PLUS overdue bucket, which indicates they are very late on payments. The job that refreshes the invoice aging mart is job_7841.

LLM judge result:
 Winner: dense
Reasoning: The dense-only answer is closer to the ground truth because it mentions "repeated overdue invoices" and "particularly those in the AR_